# CS781 Phase 1 — Model 3 (GQA + RetNet mix) — Training

**Inputs to attach** (Add Data): (1) the tokenizer dataset, (2) the tokenization notebook's output (`shards/`, `val_tokens.npy`, `tokenized_meta.json`), and, from the second session on, (3) the previous version of *this* notebook's output.

9 layers: GQA at 1, 5, 9 (reusing model 1's tested attention module directly), RetNet at 2, 3, 4, 6, 7, 8 -- a 1:2 ratio (not 1:3; flagged when this layout was chosen -- first/middle/last at 9 layers can't also hit exactly 1:3). Dense SwiGLU FFN throughout (MoE deferred to a later ablation). Reuses model 1 and model 2's exact data pipeline and calibration/resume machinery -- only the model definition and the warmup computation differ.

**`WARMUP_STEPS` is computed from `TOTAL_TRAIN_STEPS`, not hardcoded** -- this is the direct fix for the mistake that cost real GPU-hours on model 2 (a warmup value copied from a different model's step count, becoming 64% of this one's much shorter run). Making it a fraction structurally prevents that mismatch from happening again.

Needs no internet and no Hugging Face login.

## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
N_GPU = max(1, torch.cuda.device_count())

## 2. Configuration

In [ ]:
import os, json, glob, time, math, shutil
import numpy as np

SESSION_START = time.time()

# ===== change these per run =====
STEPS_THIS_RUN = 2      # optimizer steps for THIS session. 2 = quick timing/diagnostic check. None = run until the
                        # time limit below (or the end of training), whichever comes first
TOKEN_BUDGET = 130_000_000  # training tokens to use. Calibration: small. Real run: set from the printed suggestion
MAX_SESSION_HOURS = 11.0  # training stops + checkpoints after this long, so one commit always ends
                          # cleanly under the 12h limit (leaves time for final eval + save)

# ===== fixed for the whole training: do NOT change between sessions =====
SEED = 42
BLOCK = 1024                      # same as model 1 -- this architecture has no long-context
                                  # rationale the way DSA's sparsity needed model 2's 4096
N_EMBD, N_LAYER = 1024, 9
GQA_LAYERS = (0, 4, 8)             # 0-indexed: layers 1, 5, 9. Rest are RetNet.
N_HEAD, N_KV_HEAD, HEAD_DIM = 16, 4, 64    # GQA layers (matches model 1 exactly)
RETNET_HEADS = 16                          # RetNet layers (same head_dim=64, for consistency)
RETNET_CHUNK_SIZE = 128                    # chunkwise form's chunk size -- a performance knob only,
                                           # verified not to change the output (diff ~1e-7 across
                                           # very different chunk sizes on the same input)
D_FF = 2688                                # dense SwiGLU, matches model 1's FFN size

# starting guess only, corrected by the calibration run's diagnostics -- model 1's proven
# values are the closer starting point here (same BLOCK, similar attention-cost profile)
# than model 2's, which were tuned for MLA/DSA's very different memory shape
PER_DEVICE_BATCH = 8
GRAD_ACCUM_STEPS = 32              # 8 seqs x 2 GPUs x 32 = 512 seqs/step, same global batch as model 1
LEARNING_RATE = 3e-4
WARMUP_FRACTION = 0.1              # WARMUP_STEPS is computed from this x TOTAL_TRAIN_STEPS once
                                   # the data-mixing cell below determines the real total -- see
                                   # the note at the top of this notebook for why this is a
                                   # fraction and not a hardcoded step count
SAVE_STEPS = 100                  # checkpoint every N optimizer steps
EVAL_STEPS = 200

# Same escape hatch as model 2, for the same reason -- a one-time fix if the warmup fraction
# itself ever turns out wrong mid-run. Leave off for all normal sessions; if ever turned on,
# set back to False for every session after the one that uses it.
RESET_SCHEDULE_FROM_CHECKPOINT = False

# ---- data mix (fixed for the whole training; SAME source doc counts as models 1 and 2) ----
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]   # the order the tokenization notebook read the files in
SOURCE_DOCS = {"acts": 18_826, "high_courts": 2_536_738, "supreme_court": 55_549, "tribunals": 869_132}
MIX = "equal"                     # "equal" | "proportional" | {"acts": 1, "high_courts": 2, ...} (relative weights)
VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS = 25, 10, 8   # per source: 25 runs x 10 blocks held out, margin kept out of training
TOTAL_GPU_HOUR_BUDGET = 30.0
TRAIN_BUDGET_HOURS = 28.0         # plan the whole run to fit this, leaving slack under the 30h

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model3"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-mixed-final")
TRAINING_META_PATH = os.path.join(OUTPUT_DIR, "training_meta.json")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

## 3. Load Tokenizer and Token Shards

Same tokenizer and same token shards as models 1 and 2 -- attach both the tokenizer dataset and the tokenization notebook's output here. Inputs are located by searching `/kaggle/input`, so the exact mount path doesn't matter.

In [ ]:
from transformers import AutoTokenizer

def show_inputs():
    print("Attached inputs under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("  " * depth + (os.path.basename(root) or "input") + "/", f"({len(files)} files)" if files else "")
        if depth >= 3:
            dirs[:] = []

def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-mixed-final" not in d]
if not cands:
    show_inputs()
    raise FileNotFoundError("No tokenizer found. Attach your tokenizer DATASET to this notebook (Add Input); "
                            "the listing above shows what is currently attached.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

meta_dirs = find_dirs("tokenized_meta.json")
if not meta_dirs:
    show_inputs()
    raise FileNotFoundError("No tokenized_meta.json found. Attach the OUTPUT of your tokenization notebook to this notebook.")
TOKEN_ROOT = meta_dirs[0]
META_PATH = os.path.join(TOKEN_ROOT, "tokenized_meta.json")
meta = json.load(open(META_PATH))
assert meta["vocab_size"] == len(tokenizer), "token shards were built with a different tokenizer!"
print(f"Token data: {TOKEN_ROOT} | {meta['train_tokens']:,} train tokens in {len(meta['shards'])} shards")

## 3b. Build the Training Mix and Validation Set

Identical logic to models 1 and 2's mix-building cell (same corpus, same equal-mixture-per-source approach) -- only `BLOCK` differs, which just changes how the SAME flat token shards get reshaped into sequences.

In [ ]:
import torch

# ---- 1. open every shard (memory-mapped: nothing is loaded into RAM) ----
arrays = [np.load(os.path.join(TOKEN_ROOT, "shards", s), mmap_mode="r") for s in meta["shards"]]
eos = tokenizer.eos_token_id
shard_len = np.array([len(a) for a in arrays])
tok_off = np.concatenate([[0], np.cumsum(shard_len)])          # global token offset where each shard starts
n_blk = shard_len // BLOCK
cum_blk = np.concatenate([[0], np.cumsum(n_blk)])              # global id of each shard's first block
N_BLOCKS = int(cum_blk[-1])

# ---- 2. where does each source start? (count end-of-document tokens) ----
t_scan = time.time()
assert sum(SOURCE_DOCS.values()) == meta["num_docs"], "SOURCE_DOCS doesn't add up to the tokenized corpus"
val_arr = np.load(os.path.join(TOKEN_ROOT, "val_tokens.npy"))
n_val_docs = int((val_arr == eos).sum()) + (0 if int(val_arr[-1]) == eos else 1)   # tokenization sent every 20th doc to val until full
eos_per_shard = np.array([int(np.count_nonzero(a == eos)) for a in arrays])
cum_eos = np.concatenate([[0], np.cumsum(eos_per_shard)])
assert int(cum_eos[-1]) == meta["num_docs"] - n_val_docs, (
    f"doc count mismatch: {int(cum_eos[-1]):,} end-of-doc tokens in train shards vs "
    f"{meta['num_docs']:,} docs - {n_val_docs:,} val docs. The token data isn't what this cell assumes.")

def train_doc_index(g):
    # global doc index -> index among train docs (val docs were global indices 0, 20, 40, ... n_val_docs of them)
    return g - min(n_val_docs, -(-g // 20))

def token_offset_of_train_doc(t):
    # global token offset where the t-th train doc starts = just after the t-th end-of-document token
    if t == 0:
        return 0
    j = int(np.searchsorted(cum_eos, t, side="left")) - 1
    pos = np.flatnonzero(arrays[j] == eos)
    return int(tok_off[j] + pos[t - int(cum_eos[j]) - 1] + 1)

doc_edges = np.cumsum([0] + [SOURCE_DOCS[s] for s in SOURCES])
src_start_tok = np.array([0] + [token_offset_of_train_doc(train_doc_index(int(g))) for g in doc_edges[1:-1]])

# ---- 3. which source does every block belong to? ----
shard_of_blk = np.repeat(np.arange(len(arrays)), n_blk)
blk_start_tok = tok_off[shard_of_blk] + (np.arange(N_BLOCKS) - cum_blk[shard_of_blk]) * BLOCK
blk_source = np.searchsorted(src_start_tok[1:], blk_start_tok, side="right")     # 0..len(SOURCES)-1, non-decreasing
src_lo = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="left")
src_hi = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="right")

# ---- 4. hold out validation: evenly spread random runs per source, with a margin excluded from training ----
rng = np.random.default_rng(SEED)
excluded = np.zeros(N_BLOCKS, dtype=bool)
val_ids = {}
for si, s in enumerate(SOURCES):
    lo, hi = int(src_lo[si]), int(src_hi[si])
    stratum = (hi - lo) // VAL_RUNS
    assert stratum > VAL_RUN_BLOCKS + 2 * VAL_MARGIN_BLOCKS + 1, f"source {s} is too small for {VAL_RUNS} validation runs"
    runs = []
    for r in range(VAL_RUNS):
        a = lo + r * stratum + VAL_MARGIN_BLOCKS + int(rng.integers(0, stratum - VAL_RUN_BLOCKS - 2 * VAL_MARGIN_BLOCKS))
        runs.append(np.arange(a, a + VAL_RUN_BLOCKS))
        excluded[a - VAL_MARGIN_BLOCKS: a + VAL_RUN_BLOCKS + VAL_MARGIN_BLOCKS] = True
    val_ids[s] = np.concatenate(runs)

# ---- 5. how many blocks does each source contribute? ----
cand = {s: np.arange(src_lo[i], src_hi[i])[~excluded[src_lo[i]:src_hi[i]]] for i, s in enumerate(SOURCES)}
avail = {s: len(cand[s]) for s in SOURCES}

def allocate(avail, weights, total):
    # split `total` blocks across sources in proportion to `weights`, never exceeding what a source has;
    # whatever a small source can't absorb is redistributed over the others
    alloc = {s: 0 for s in avail}
    active = [s for s in avail if weights[s] > 0]
    remaining = int(total)
    while active and remaining > 0:
        wsum = sum(weights[s] for s in active)
        share = {s: remaining * weights[s] / wsum for s in active}
        capped = [s for s in active if share[s] >= avail[s]]
        if not capped:
            for s in active:
                alloc[s] = int(share[s])
            break
        for s in capped:
            alloc[s] = avail[s]
            remaining -= avail[s]
            active.remove(s)
    return alloc

if MIX == "equal":
    weights = {s: 1.0 for s in SOURCES}
elif MIX == "proportional":
    weights = {s: float(avail[s]) for s in SOURCES}
else:
    weights = {s: float(MIX[s]) for s in SOURCES}
alloc = allocate(avail, weights, TOKEN_BUDGET // BLOCK)
train_ids = rng.permutation(np.concatenate([rng.choice(cand[s], size=alloc[s], replace=False) for s in SOURCES]))

# ---- 6. datasets ----
class PickedBlocks(torch.utils.data.Dataset):
    """Fixed-length blocks, read on demand from the memory-mapped shards by global block id."""
    def __init__(self, ids):
        self.ids = np.asarray(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        g = int(self.ids[i])
        j = int(np.searchsorted(cum_blk, g, side="right") - 1)
        k = g - int(cum_blk[j])
        return {"input_ids": arrays[j][k * BLOCK:(k + 1) * BLOCK].astype(np.int64).tolist()}

train_dataset = PickedBlocks(train_ids)
val_by_source = {s: PickedBlocks(val_ids[s]) for s in SOURCES}
val_dataset = PickedBlocks(np.concatenate([val_ids[s] for s in SOURCES]))

GLOBAL_SEQS = PER_DEVICE_BATCH * N_GPU * GRAD_ACCUM_STEPS
TOTAL_TRAIN_STEPS = len(train_dataset) // GLOBAL_SEQS          # one pass over the data
WARMUP_STEPS = max(10, int(WARMUP_FRACTION * TOTAL_TRAIN_STEPS))   # derived, not hardcoded -- see cell 2's note

print(f"scanned {len(arrays)} shards in {time.time() - t_scan:.0f}s | {meta['num_docs']:,} docs, {n_val_docs:,} of them were held out by the tokenizer")
print(f"MIX = {MIX!r} | budget {TOKEN_BUDGET/1e6:,.0f}M tokens (corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"{'source':14s}{'available':>11s}{'used':>10s}{'share':>8s}{'of source':>11s}")
_tot = max(sum(alloc.values()), 1)
for s in SOURCES:
    print(f"{s:14s}{avail[s] * BLOCK / 1e6:10.0f}M{alloc[s] * BLOCK / 1e6:9.0f}M{100 * alloc[s] / _tot:7.1f}%{100 * alloc[s] / max(avail[s], 1):10.1f}%")
print(f"validation: {len(val_dataset):,} blocks ({VAL_RUNS * VAL_RUN_BLOCKS} per source, in {VAL_RUNS} runs of {VAL_RUN_BLOCKS})")
print(f"Train blocks: {len(train_dataset):,} | Tokens/step: {GLOBAL_SEQS * BLOCK:,} | TOTAL_TRAIN_STEPS (one epoch): {TOTAL_TRAIN_STEPS:,}")
print(f"WARMUP_STEPS = {WARMUP_STEPS:,} ({100*WARMUP_FRACTION:.0f}% of total, computed from the real step count above)")
if TOKEN_BUDGET < 1_000_000_000:
    print("NOTE: small TOKEN_BUDGET -> calibration only. Set it from the printed suggestion for the real run.")

## 4. Define The Model

Three files: `legal_gqa_model.py` (model 1's tested file, unchanged -- supplies RMSNorm, GroupedQueryAttention, SwiGLU, reused directly for the GQA layers and the dense FFN), `retnet.py` (Multi-Scale Retention: parallel form for training, recurrent form for generation -- both verified exact duals of each other in testing, including an exhaustive causal-safety sweep and a parallel-prefill-then-recurrent-continuation check), `model.py` (the mixed decoder stack, with per-layer-type dispatch and a uniform cache convention so GQA's (K,V) cache and RetNet's fixed-size state don't need special-cased handling). Written to real files and imported for the same reason as models 1 and 2: `transformers` reads `__file__` off a model class's module.

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
%%writefile /kaggle/working/retnet.py
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    return (x * cos) + (rotate_half(x) * sin)


def rope_table(T, dim, theta, device, dtype):
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    pos = torch.arange(T, device=device).float()
    emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
    return emb.cos().to(dtype), emb.sin().to(dtype)


def retention_decay_rates(n_heads, device, dtype):
    """gamma_h = 1 - 2^-(5 + 3h/(H-1)), h=0..H-1 (Sun et al. 2023, confirmed against a
    second independent source). Spans a fixed range (half-life ~22 to ~177 tokens)
    regardless of head count -- NOT the same as naively doing 2^-(5+h), which would
    make the slowest head's decay depend on how many heads there are."""
    if n_heads == 1:
        exponents = torch.tensor([5.0])
    else:
        h = torch.arange(n_heads, device=device).float()
        exponents = 5.0 + 3.0 * h / (n_heads - 1)
    return (1.0 - 2.0 ** (-exponents)).to(device=device, dtype=dtype)


class MultiScaleRetention(nn.Module):
    """Sun et al. 2023. Replaces softmax attention entirely -- no QK-softmax
    anywhere. Parallel form for training, recurrent form (fixed-size state,
    not a growing KV cache) for generation; both are exact mathematical duals
    of the same operator, verified in testing, not just structurally similar."""
    def __init__(self, d_model, n_heads, dropout=0.0, rope_theta=10000.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.rope_theta = rope_theta
        self.dropout = dropout

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_G = nn.Linear(d_model, d_model, bias=False)   # swish-gate pathway
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.group_norm = nn.GroupNorm(n_heads, d_model)

    def _project(self, x, past_len=0):
        B, T, _ = x.shape
        H, Dh = self.n_heads, self.head_dim
        Q = self.W_Q(x).view(B, T, H, Dh).transpose(1, 2)
        K = self.W_K(x).view(B, T, H, Dh).transpose(1, 2)
        V = self.W_V(x).view(B, T, H, Dh).transpose(1, 2)
        cos, sin = rope_table(past_len + T, Dh, self.rope_theta, x.device, x.dtype)
        cos, sin = cos[past_len:past_len + T], sin[past_len:past_len + T]
        Q = apply_rope(Q, cos, sin)
        K = apply_rope(K, cos, sin)
        return Q, K, V

    def forward_parallel(self, x):
        """Training / full-sequence forward. O(T^2) in memory like dense
        attention, which is fine at our context length (already established
        for model 1's dense attention at the same length)."""
        B, T, D = x.shape
        Q, K, V = self._project(x)
        gamma = retention_decay_rates(self.n_heads, x.device, torch.float32)  # decay math in fp32: stays stable even under fp16 autocast

        n = torch.arange(T, device=x.device)
        rel = (n[:, None] - n[None, :]).float()                 # (T,T), n-m
        causal = rel >= 0
        # gamma[:,None,None]**rel broadcasts to (H,T,T); clamp the exponent
        # where causal is False so an already-negative rel never produces a
        # huge value from **-x before the mask zeroes it out
        rel_clamped = rel.clamp(min=0)
        D_mat = (gamma[:, None, None] ** rel_clamped[None]) * causal[None].to(torch.float32)  # (H,T,T)

        # Q,K cast to fp32 BEFORE the matmul, not after -- reverting an earlier speed
        # attempt that computed this in native fp16 for tensor cores. That reasoning
        # (fp16 matmul accumulates in fp32 internally, so no precision is lost) is true
        # for standard attention but doesn't carry over here: softmax immediately bounds
        # attention's raw QK^T scores via its max-subtraction trick, so individual
        # products overflowing fp16 isn't a real risk there. Retention has no such
        # nonlinearity -- confirmed directly: at head_dim=64 (this model's real size)
        # and Q/K magnitudes plausible after real training (not just fresh-init scale,
        # which is all my original tests exercised), the fp16 matmul itself overflows to
        # inf BEFORE any later .float() cast can rescue it -- this is the actual
        # mechanism behind a real NaN-loss failure partway through real training.
        scores = torch.einsum("bhtd,bhsd->bhts", Q.float(), K.float())          # (B,H,T,T)
        R = scores * D_mat[None]
        # Row-normalization from the paper: bounds retention's output scale the
        # way softmax naturally does for attention -- without this, heads with
        # very different decay rates produce very different output magnitudes.
        row_norm = R.abs().sum(dim=-1, keepdim=True).clamp(min=1.0)
        R = R / row_norm
        out = torch.einsum("bhts,bhsd->bhtd", R.to(V.dtype), V)

        out = out.transpose(1, 2).contiguous().view(B, T, D)
        # GroupNorm must treat each (batch, timestep) pair independently -- nn.GroupNorm
        # on a (B,D,T) tensor pools its statistics across the WHOLE T dimension jointly
        # with the channels, which leaks future positions into earlier ones through the
        # shared mean/var (confirmed: this was an actual causal-safety failure in testing,
        # not just a theoretical concern). Folding B and T into one axis first makes each
        # token's normalization independent of every other token's, the same way LayerNorm
        # is independent per-token in a normal transformer.
        out = self.group_norm(out.reshape(B * T, D)).reshape(B, T, D)
        gate = F.silu(self.W_G(x))
        out = gate * out
        return self.W_O(out), None

    def prefill_state(self, x):
        """The state forward_recurrent would have accumulated after processing
        x as a sequence of single-token steps from scratch, computed directly
        (not by actually looping) so a multi-token prompt can be prefilled with
        the efficient parallel form and then continued one token at a time."""
        T = x.shape[1]
        Q, K, V = self._project(x)
        gamma = retention_decay_rates(self.n_heads, x.device, torch.float32)  # (H,)
        n = torch.arange(T, device=x.device).float()
        weight = gamma[:, None] ** (T - 1 - n)[None, :]                # (H,T): gamma^(T-1-m)
        state = torch.einsum("bhmd,bhme,hm->bhde", K.float(), V.float(), weight)
        return state

    def forward_recurrent(self, x, past_state=None, past_len=0):
        """Single-token generation step. past_state: (B,H,head_dim,head_dim)
        per layer, NOT a growing cache -- this is RetNet's whole efficiency
        point, fixed-size regardless of how long generation runs. past_len is
        this token's absolute position (needed for correct RoPE rotation,
        same role it plays in MLA's up_project)."""
        B, T, D = x.shape
        assert T == 1, "recurrent form is for single-token decode steps"
        Q, K, V = self._project(x, past_len=past_len)
        gamma = retention_decay_rates(self.n_heads, x.device, torch.float32).view(1, self.n_heads, 1, 1)

        kv = torch.einsum("bhtd,bhte->bhde", K.float(), V.float())  # outer product, (B,H,head_dim,head_dim)
        if past_state is None:
            state = kv
        else:
            state = gamma * past_state + kv

        out = torch.einsum("bhtd,bhde->bhte", Q.float(), state)     # (B,H,1,head_dim)
        out = out.to(V.dtype).transpose(1, 2).contiguous().view(B, T, D)
        out = self.group_norm(out.reshape(B * T, D)).reshape(B, T, D)  # same per-token GroupNorm as forward_parallel
        gate = F.silu(self.W_G(x))
        out = gate * out
        return self.W_O(out), state

    def forward_chunkwise(self, x, chunk_size=128):
        """Hybrid form: parallel math within each chunk, recurrent state carried
        between chunks -- avoids ever materializing the full (T,T) matrix that
        made forward_parallel memory-bandwidth-bound (confirmed: per-layer cost
        was 4-5x GQA's, GPU util near-idle, matching a bandwidth-bound not a
        compute-bound signature). Mathematically exact, not an approximation --
        verified against both other forms in testing, not just asserted."""
        B, T, D = x.shape
        H, Dh = self.n_heads, self.head_dim
        Q, K, V = self._project(x)
        gamma = retention_decay_rates(self.n_heads, x.device, torch.float32)

        state = torch.zeros(B, H, Dh, Dh, device=x.device, dtype=torch.float32)
        outputs = []
        for c0 in range(0, T, chunk_size):
            c1 = min(c0 + chunk_size, T)
            C = c1 - c0
            Qc, Kc, Vc = Q[:, :, c0:c1], K[:, :, c0:c1], V[:, :, c0:c1]
            local = torch.arange(C, device=x.device).float()

            # inner-chunk: identical math to forward_parallel, just on this chunk.
            # No row-normalization here (unlike forward_parallel) -- it would need
            # to apply to the INNER+CROSS sum together to stay a pure no-op after
            # GroupNorm (per the scale-invariance argument that justified it in
            # forward_parallel); applying it to only the inner term would unevenly
            # rescale inner vs cross before they're summed, a real change in value,
            # not just a numerically-safer route to the same one. Chunks are far
            # shorter than the full sequence, so the underflow risk it guarded
            # against is correspondingly smaller -- fp32 alone covers it here.
            rel = (local[:, None] - local[None, :])
            causal = rel >= 0
            D_mat = (gamma[:, None, None] ** rel.clamp(min=0)[None]) * causal[None].to(torch.float32)
            scores = torch.einsum("bhtd,bhsd->bhts", Qc.float(), Kc.float())  # fp32 before the matmul -- same fix as forward_parallel, same reason
            # V-matmul also stays in fp32 all the way through -- unlike forward_parallel,
            # this term has no row-normalization bounding it before a cast to fp16 would
            # happen (removed deliberately, see the note above on why), so there was
            # nothing protecting an fp32->fp16 downcast here from overflowing on its own
            # even with the QK^T matmul itself already fixed. Matches how the cross-chunk
            # term below already (correctly) stays fp32 until the final combine.
            inner = torch.einsum("bhts,bhsd->bhtd", scores * D_mat[None], Vc.float())

            # cross-chunk: each position queries the incoming state, scaled by
            # how far that position sits from the start of this chunk
            xi = gamma[:, None] ** (local[None, :] + 1)                  # (H,C)
            cross = torch.einsum("bhtd,bhde->bhte", Qc.float(), state) * xi[None, :, :, None]

            outputs.append((inner + cross).to(Vc.dtype))  # both already fp32 at this point

            # carry state forward: decay the old state across this whole chunk,
            # add this chunk's own (zero-start) contribution -- same computation
            # prefill_state already does and already passed testing, just scoped
            # to one chunk with a nonzero carry-in added on top
            zeta = gamma[:, None] ** (C - 1 - local[None, :])            # (H,C)
            state_contrib = torch.einsum("bhmd,bhme,hm->bhde", Kc.float(), Vc.float(), zeta)
            state = (gamma ** C)[None, :, None, None] * state + state_contrib

        out = torch.cat(outputs, dim=2).transpose(1, 2).contiguous().view(B, T, D)
        out = self.group_norm(out.reshape(B * T, D)).reshape(B, T, D)
        gate = F.silu(self.W_G(x))
        out = gate * out
        return self.W_O(out), state

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from legal_gqa_model import RMSNorm, GroupedQueryAttention, SwiGLU
from retnet import MultiScaleRetention, rope_table as gqa_rope_table


class LegalMixedConfig(PretrainedConfig):
    model_type = "legal_mixed_gqa_retnet"

    def __init__(
        self,
        vocab_size=50257, n_positions=1024, n_embd=1024, n_layer=9,
        gqa_layers=(0, 4, 8),           # 0-indexed: layers 1, 5, 9 in 1-indexed terms
        n_head=16, n_kv_head=4,         # GQA layers
        retnet_heads=16,                # RetNet layers
        retnet_chunk_size=128,          # chunkwise form's chunk size -- a performance knob
                                        # only, verified to not change the output (tested
                                        # chunk_size=5 vs chunk_size=50 on the same input: diff
                                        # ~1e-7, floating-point noise, not a real difference)
        d_ff=2688, rope_theta=10000.0,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0 and n_head % n_kv_head == 0
        assert n_embd % retnet_heads == 0
        self.vocab_size, self.n_positions, self.n_embd, self.n_layer = vocab_size, n_positions, n_embd, n_layer
        self.gqa_layers = list(gqa_layers)
        self.n_head, self.n_kv_head, self.retnet_heads = n_head, n_kv_head, retnet_heads
        self.retnet_chunk_size = retnet_chunk_size
        self.d_ff, self.rope_theta = d_ff, rope_theta
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class MixedDecoderBlock(nn.Module):
    def __init__(self, config: LegalMixedConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_gqa = layer_idx in config.gqa_layers
        self.retnet_chunk_size = config.retnet_chunk_size
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        if self.is_gqa:
            self.attn = GroupedQueryAttention(config)
        else:
            self.attn = MultiScaleRetention(config.n_embd, config.retnet_heads, config.dropout, config.rope_theta)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_kv=None, use_cache=False, past_len=0):
        xn = self.ln_1(x)
        if self.is_gqa:
            attn_out, present = self.attn(xn, cos, sin, attn_bias, is_causal, past_kv, use_cache)
        else:
            T = xn.shape[1]
            if T == 1:
                attn_out, present = self.attn.forward_recurrent(xn, past_state=past_kv, past_len=past_len)
            else:
                # chunkwise form, not forward_parallel -- never materializes the full
                # (T,T) matrix that made forward_parallel memory-bandwidth-bound (measured
                # 4-5x GQA's per-layer cost; verified exact, not an approximation, against
                # both other forms before this switch). Its returned state already matches
                # what prefill_state/forward_recurrent would give (verified to 3e-6), so it
                # covers prefill for generation too -- no separate prefill_state call needed.
                attn_out, present = self.attn.forward_chunkwise(xn, chunk_size=self.retnet_chunk_size)
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalMixedForCausalLM(PreTrainedModel):
    config_class = LegalMixedConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalMixedConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([MixedDecoderBlock(config, i) for i in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "W_O.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Only used for GQA layers -- RetNet's decay matrix already handles
        causality on its own and needs no external mask."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1
        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min
        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None]
        return bias, False

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # no **kwargs above on purpose -- see model 1/2's build notes: Trainer treats a
        # forward() accepting **kwargs as self-normalizing its loss, breaking grad-accum scaling

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        B, T = input_ids.shape
        # Every cache entry is (data, position_after_this_entry), uniformly for GQA's
        # (K,V) tuple and RetNet's fixed-size state alike -- RetNet's state has no T
        # dimension to read a length back out of, so this avoids needing different
        # logic per layer type just to recover how far into the sequence we are.
        past_len = past_key_values[0][1] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = gqa_rope_table(past_len + T, self.head_dim, self.config.rope_theta, input_ids.device, x.dtype)
        cos, sin = cos[past_len:past_len + T], sin[past_len:past_len + T]
        attn_bias, is_causal = self._build_attn_bias(attention_mask, past_len, T, input_ids.device, x.dtype)

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            block_pkv = past_key_values[i][0] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block, pkv=block_pkv, pl=past_len):
                    h2, _ = blk(h, cos, sin, attn_bias, is_causal, pkv, False, pl)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, block_pkv, use_cache, past_len)
                if use_cache:
                    presents.append((present, past_len + T))

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from model import LegalMixedConfig, LegalMixedForCausalLM

config = LegalMixedConfig(
    vocab_size=len(tokenizer), n_positions=BLOCK,
    n_embd=N_EMBD, n_layer=N_LAYER, gqa_layers=GQA_LAYERS,
    n_head=N_HEAD, n_kv_head=N_KV_HEAD, retnet_heads=RETNET_HEADS, retnet_chunk_size=RETNET_CHUNK_SIZE, d_ff=D_FF,
    bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    activation_checkpointing=True,
)
model = LegalMixedForCausalLM(config)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print(f"GQA layers (0-indexed): {GQA_LAYERS} | RetNet layers: {[i for i in range(N_LAYER) if i not in GQA_LAYERS]}")

## 5. Training Arguments

`max_steps` is the **whole** training length, so the cosine schedule is one continuous curve across sessions. `StopAfter` ends *this session* early and forces a checkpoint.

In [ ]:
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          DataCollatorForLanguageModeling)
from transformers.trainer_utils import get_last_checkpoint

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    max_steps=TOTAL_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.1,
    adam_beta1=0.9, adam_beta2=0.95,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    fp16=torch.cuda.is_available(),      # T4 is Turing: fp16, not bf16
    max_grad_norm=1.0,
    eval_strategy="steps", eval_steps=EVAL_STEPS, prediction_loss_only=True,
    logging_steps=10,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    dataloader_num_workers=2,
    seed=SEED,
    report_to="none",
    disable_tqdm=True,   # the notebook progress widget has a bug that crashes a session ending at exactly 2 steps; plain-text logs suit a commit anyway
)

class StopAfter(TrainerCallback):
    """Ends THIS session (with a checkpoint) at a step count OR a wall-clock deadline."""
    def __init__(self, stop_step, deadline):
        self.stop_step, self.deadline = stop_step, deadline
    def on_step_end(self, args, state, control, **kwargs):
        out_of_time = time.time() > self.deadline
        if state.global_step >= self.stop_step or out_of_time:
            if out_of_time and state.global_step < self.stop_step:
                print(f"\nSession time limit reached at step {state.global_step}: checkpointing and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

class StopOnNaN(TrainerCallback):
    """Stops the moment a logged loss or grad_norm goes non-finite, instead of
    continuing to train (and checkpoint) over an already-corrupted state. Can't
    undo the step that just corrupted the weights -- that's already applied by
    the time this fires -- but bounds the damage to at most SAVE_STEPS of lost
    progress rather than silently continuing for however long until someone
    happens to notice (which is what actually made the last failure so costly:
    not that NaN occurred, but that it went undetected across several checkpoints)."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control
        bad = [k for k in ("loss", "grad_norm") if k in logs and not math.isfinite(logs[k])]
        if bad:
            print(f"\nNon-finite {bad} at step {state.global_step} ({logs}): stopping immediately, "
                  f"not continuing to train or checkpoint over a corrupted state.")
            control.should_training_stop = True
        return control

## 6. Train (auto-resume + per-step timing)

In [ ]:
# --- pull the newest checkpoint from a previously attached output, if we have none yet ---
if not glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*")):
    prev = glob.glob("/kaggle/input/**/legal-mixed-final/checkpoint-*", recursive=True)
    if prev:
        latest = max(prev, key=lambda p: int(p.rsplit("-", 1)[1]))
        shutil.copytree(latest, os.path.join(MODEL_OUTPUT_DIR, os.path.basename(latest)))
        prev_meta = os.path.join(os.path.dirname(os.path.dirname(latest)), "training_meta.json")
        if os.path.exists(prev_meta):
            shutil.copy(prev_meta, TRAINING_META_PATH)
        print("Restored checkpoint from previous output:", latest)

fingerprint = {"global_seqs": GLOBAL_SEQS, "block": BLOCK, "total_steps": TOTAL_TRAIN_STEPS,
               "arch": [N_EMBD, N_LAYER, list(GQA_LAYERS), N_HEAD, N_KV_HEAD, RETNET_HEADS, RETNET_CHUNK_SIZE, D_FF],
               "lr": LEARNING_RATE, "warmup": WARMUP_STEPS,
               "data": [TOKEN_BUDGET, str(MIX), SEED, VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS]}
cumulative_seconds = 0.0
if os.path.exists(TRAINING_META_PATH):
    saved = json.load(open(TRAINING_META_PATH))
    cumulative_seconds = saved.get("cumulative_seconds", 0.0)
    mismatch = saved.get("fingerprint") != fingerprint and glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*"))
    if mismatch and not RESET_SCHEDULE_FROM_CHECKPOINT:
        raise RuntimeError(f"Checkpoint was made with a different config!\n saved:   {saved.get('fingerprint')}\n current: {fingerprint}\n"
                           "Either restore the old settings, or set RESET_SCHEDULE_FROM_CHECKPOINT=True if this "
                           "mismatch is deliberate (a one-time schedule fix), or start fresh (do not attach the old output).")

last_checkpoint = get_last_checkpoint(MODEL_OUTPUT_DIR)
steps_at_start = int(last_checkpoint.rsplit("-", 1)[1]) if last_checkpoint else 0

if RESET_SCHEDULE_FROM_CHECKPOINT and last_checkpoint:
    print(f"RESET_SCHEDULE_FROM_CHECKPOINT: loading weights from step {steps_at_start}, "
          f"discarding the old optimizer/scheduler state, starting a fresh schedule "
          f"(warmup={WARMUP_STEPS}, total={TOTAL_TRAIN_STEPS}).")
    print("Training data will be read from the start again -- the model will see "
          f"its first ~{steps_at_start} steps'-worth of data a second time before "
          "reaching genuinely new material.")
    model = LegalMixedForCausalLM.from_pretrained(last_checkpoint)
    resume_arg = None
    stop_step = TOTAL_TRAIN_STEPS if STEPS_THIS_RUN is None else min(STEPS_THIS_RUN, TOTAL_TRAIN_STEPS)
else:
    resume_arg = last_checkpoint
    stop_step = TOTAL_TRAIN_STEPS if STEPS_THIS_RUN is None else min(steps_at_start + STEPS_THIS_RUN, TOTAL_TRAIN_STEPS)
    print(f"Resuming from step {steps_at_start}" if last_checkpoint else "Starting fresh.")

deadline = SESSION_START + MAX_SESSION_HOURS * 3600
print(f"This session runs to step {stop_step} of {TOTAL_TRAIN_STEPS} (or until {MAX_SESSION_HOURS}h of session time)")

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[StopAfter(stop_step, deadline), StopOnNaN()],
)

# ---- diagnostics: is the GPU busy (sampled continuously across the whole step,
# not a single post-hoc snapshot -- a one-shot reading can land in a brief idle
# gap between batches and understate a GPU that was actually busy the rest of
# the time), and are matmuls really running in fp16? ----
import subprocess, threading
gpu_samples = []
def _num(v):
    try: return float(v)
    except ValueError: return float("nan")
def _monitor(stop):
    while not stop.is_set():
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,power.draw,clocks.sm",
                 "--format=csv,noheader,nounits"], text=True, timeout=10)
            gpu_samples.append([[_num(v.strip()) for v in ln.split(",")] for ln in out.strip().splitlines()])
        except Exception:
            return
        stop.wait(5)
_stop = threading.Event()
threading.Thread(target=_monitor, args=(_stop,), daemon=True).start()

# hooks a Linear projection's output, matching models 1/2's proven check -- NOT a
# normalization layer: RMSNorm's own reductions can legitimately stay fp32 under
# autocast even when everything else is correctly running in fp16 (this is exactly
# the mechanism behind an earlier dtype bug found during this model's build), so
# checking a norm layer here would give a false negative, not a useful signal
_probe = {}
def _record_dtype(module, inp, out):
    _probe["q_proj output dtype"] = out.dtype
_target_mod = model.module if hasattr(model, "module") else model
_hook = _target_mod.blocks[0].attn.q_proj.register_forward_hook(_record_dtype)

t0 = time.time()
trainer.train(resume_from_checkpoint=resume_arg)
elapsed = time.time() - t0
_stop.set(); _hook.remove()

print()
print("--- diagnostics ---")
print("matmul output dtype inside the model:", _probe.get("q_proj output dtype", "n/a"), "(float16 = tensor cores in use)")

if gpu_samples:
    for g in range(len(gpu_samples[0])):
        col = lambda k: [smp[g][k] for smp in gpu_samples if len(smp) > g and smp[g][k] == smp[g][k]]
        m = lambda xs: sum(xs) / len(xs) if xs else float("nan")
        print(f"GPU {g}: util {m(col(1)):.0f}% | mem peak {max(col(2) or [0]):.0f} MB | power {m(col(3)):.0f} W | SM clock {m(col(4)):.0f} MHz  ({len(gpu_samples)} samples)")
else:
    print("nvidia-smi sampling unavailable")
if torch.cuda.is_available():
    print("torch peak allocated:", [f"{torch.cuda.max_memory_allocated(i)/2**30:.1f} GB" for i in range(torch.cuda.device_count())])

steps_done = trainer.state.global_step if RESET_SCHEDULE_FROM_CHECKPOINT else trainer.state.global_step - steps_at_start
sec_per_step = elapsed / max(steps_done, 1)
cumulative_seconds += elapsed
json.dump({"cumulative_seconds": cumulative_seconds, "global_step": trainer.state.global_step,
           "fingerprint": fingerprint}, open(TRAINING_META_PATH, "w"))

remaining_steps = TOTAL_TRAIN_STEPS - trainer.state.global_step
print()
print(f"Steps this session: {steps_done} | wall time: {elapsed/60:.1f} min | {sec_per_step:.1f} s/step")
print(f"Progress: step {trainer.state.global_step:,} / {TOTAL_TRAIN_STEPS:,}")
print(f"GPU time used so far: {cumulative_seconds/3600:.2f} / {TOTAL_GPU_HOUR_BUDGET:.0f} h")
print(f"Full remaining training at this speed: {remaining_steps * sec_per_step / 3600:.1f} h")

budget_steps = int(TRAIN_BUDGET_HOURS * 3600 / sec_per_step)
fit_tokens = budget_steps * GLOBAL_SEQS * BLOCK
print()
print(f"At {sec_per_step:.1f} s/step, {TRAIN_BUDGET_HOURS:.0f}h of training = {budget_steps:,} steps = {budget_steps * GLOBAL_SEQS * BLOCK / 1e9:.2f}B tokens")
print(f"  -> real run: set TOKEN_BUDGET = {int(fit_tokens // 10_000_000 * 10_000_000):,}  (the corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"  -> ~{math.ceil(TRAIN_BUDGET_HOURS / MAX_SESSION_HOURS)} sessions of up to {MAX_SESSION_HOURS:.0f}h; set STEPS_THIS_RUN = None and each one stops itself in time")
print(f"  -> WARMUP_FRACTION={WARMUP_FRACTION} will auto-recompute WARMUP_STEPS for whatever TOTAL_TRAIN_STEPS this new TOKEN_BUDGET gives -- no manual adjustment needed")
if steps_done <= 3:
    print("(few steps measured: first-step overhead inflates this; treat as an upper bound)")
print()
print("If this OOM'd instead of printing the above: halve PER_DEVICE_BATCH and double GRAD_ACCUM_STEPS, re-run.")

## 6b. Where Is The Time Going? (GQA vs RetNet layer cost)

Same approach as model 2's breakdown: a single un-parallelized copy of the model on one GPU, not the `DataParallel`-wrapped training model. Reports GQA and RetNet layer time SEPARATELY -- directly useful for the report, since this is the one place the two mechanisms run side by side in the same model under the same conditions.

In [ ]:
if torch.cuda.is_available():
    probe = LegalMixedForCausalLM(config).to("cuda:0").eval()
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    probe.load_state_dict(state)

    loader = torch.utils.data.DataLoader(train_dataset, batch_size=PER_DEVICE_BATCH, collate_fn=data_collator)
    batch = next(iter(loader))
    input_ids = batch["input_ids"].to("cuda:0")

    def timed(fn, warmup=1, reps=3):
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(reps):
            result = fn()
        torch.cuda.synchronize()
        return (time.time() - t0) / reps, result

    times = {}
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        times["embedding"], x = timed(lambda: probe.drop(probe.wte(input_ids)))
        cos, sin = None, None
        from retnet import rope_table as _rope_table
        cos, sin = _rope_table(x.shape[1], probe.head_dim, config.rope_theta, x.device, x.dtype)

        for i, block in enumerate(probe.blocks):
            ln1_time, xn = timed(lambda: block.ln_1(x))
            times.setdefault("ln_1", 0.0); times["ln_1"] += ln1_time

            label = "gqa_attn" if block.is_gqa else "retnet_attn"
            if block.is_gqa:
                attn_time, (attn_out, _) = timed(lambda: block.attn(xn, cos, sin, None, True, None, False))
            else:
                attn_time, (attn_out, _) = timed(lambda: block.attn.forward_chunkwise(xn, chunk_size=RETNET_CHUNK_SIZE))
            times.setdefault(label, 0.0); times[label] += attn_time
            times.setdefault(label + "_layers", 0); times[label + "_layers"] += 1

            x2 = x + attn_out
            ln2_time, xn2 = timed(lambda: block.ln_2(x2))
            times.setdefault("ln_2", 0.0); times["ln_2"] += ln2_time

            ffn_time, ffn_out = timed(lambda: block.ffn(xn2))
            times.setdefault("ffn", 0.0); times["ffn"] += ffn_time

            x = x2 + ffn_out

    layer_counts = {k: times.pop(k) for k in list(times) if k.endswith("_layers")}
    total = sum(times.values())
    print(f"Per-component time (one forward pass, {len(probe.blocks)} layers, single GPU, batch={PER_DEVICE_BATCH}):")
    for k, v in sorted(times.items(), key=lambda kv: -kv[1]):
        n = layer_counts.get(k + "_layers")
        suffix = f"  ({n} layers, {v/n*1000:.1f}ms/layer)" if n else ""
        print(f"  {k:24s} {v*1000:8.1f} ms  ({100*v/total:5.1f}%){suffix}")
    print(f"  {'TOTAL':24s} {total*1000:8.1f} ms")
    del probe
    torch.cuda.empty_cache()
else:
    print("No GPU available -- skipping component timing breakdown.")

## 7. Validation Loss, Loss Plot, Save Model

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

eval_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss: {eval_loss:.4f} | perplexity: {math.exp(eval_loss):.2f}")

logs = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(9, 4.5))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), label="val loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss.png"), dpi=150)

print("Per-source validation:")
for s in SOURCES:
    r = trainer.evaluate(eval_dataset=val_by_source[s], metric_key_prefix=f"val_{s}")
    print(f"  {s:14s} loss {r[f'val_{s}_loss']:.4f} | perplexity {math.exp(r[f'val_{s}_loss']):.1f}")

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Saved model + tokenizer to", MODEL_OUTPUT_DIR)